## Healthcare Nursing Documentation Burden — Fabric IQ

### Create Ontology from Package (`.iq` file)

This notebook creates the **NursingDocBurdenOntology** in Fabric IQ using the `.iq` ontology package.
It binds entity types to Lakehouse Delta tables and contextualizations to Eventhouse KQL tables.

**Ontology Structure:**
- 6 Entity Types: Nurse, Patient, HospitalUnit, DocumentationType, Medication, Diagnosis
- 5 Relationship Types: assigned_to, admitted_to, has_diagnosis, cares_for, prescribes_for
- 6 Contextualizations: DocumentationEvent, MedicationAdministration, VitalSignsRecording, AssessmentEvent, EHRInteraction, HandoffReport

> **Prerequisites:**
> 1. Run `Healthcare_Generate_Ontology_Data.ipynb` first to populate Lakehouse & Eventhouse
> 2. Upload `nursing_doc_burden_ontology_package.iq` to `/lakehouse/default/Files/`
> 3. Fill in binding names below (Lakehouse name, Eventhouse name & cluster URI)

In [ ]:
# Install Fabric IQ Ontology Accelerator Package
%pip install /lakehouse/default/Files/fabriciq_ontology_accelerator-0.1.0-py3-none-any.whl --q

In [ ]:
import sempy.fabric as fabric
import json
from fabricontology import create_ontology_item, generate_definition_from_package

workspace_id = fabric.get_workspace_id()
access_token = notebookutils.credentials.getToken('pbi')

# ── Ontology Configuration ──────────────────────────────────────────────
ontology_item_name = "NursingDocBurdenOntology"
ontology_package_path = "/lakehouse/default/Files/healthcare_nursing_ontology_package.iq"

# ── Binding Configuration ───────────────────────────────────────────────
binding_lakehouse_name = "HealthcareLakehouse"       # Lakehouse with dim tables as Delta
binding_lakehouse_schema_name = "dbo"
binding_eventhouse_name = "medical_data_rt_store"     # Eventhouse with KQL event tables
binding_eventhouse_cluster_uri = "https://<YOUR_EVENTHOUSE>.z<N>.kusto.fabric.microsoft.com"
binding_eventhouse_database_name = "medical_data_rt_store"

# ── Resolve item IDs from Fabric workspace ─────────────────────────────
items_df = fabric.list_items()
binding_lakehouse_item_id = str(
    items_df[(items_df["Type"] == "Lakehouse") & (items_df["Display Name"] == binding_lakehouse_name)].iloc[0].Id
)
binding_eventhouse_item_id = str(
    items_df[(items_df["Type"] == "Eventhouse") & (items_df["Display Name"] == binding_eventhouse_name)].iloc[0].Id
)
binding_workspace_id = workspace_id

print(f"Workspace ID:  {workspace_id}")
print(f"Lakehouse ID:  {binding_lakehouse_item_id}")
print(f"Eventhouse ID: {binding_eventhouse_item_id}")

# ── Generate ontology definition from .iq package ──────────────────────
ontology_definition, entity_types, relationship_types, data_bindings, contextualizations = generate_definition_from_package(
    ontology_package_path=ontology_package_path,
    ontology_name=ontology_item_name,
    binding_workspace_id=binding_workspace_id,
    binding_lakehouse_item_id=binding_lakehouse_item_id,
    binding_lakehouse_schema_name=binding_lakehouse_schema_name,
    binding_eventhouse_item_id=binding_eventhouse_item_id,
    binding_eventhouse_cluster_uri=binding_eventhouse_cluster_uri,
    binding_eventhouse_database_name=binding_eventhouse_database_name,
)

print(f"\nEntity Types ({len(entity_types)}): {list(entity_types.keys())}")
print(f"Relationship Types ({len(relationship_types)}): {list(relationship_types.keys())}")
print(f"Contextualizations ({len(contextualizations)}): {list(contextualizations.keys())}")
print(f"Data Bindings ({len(data_bindings)}): {list(data_bindings.keys())}")

In [ ]:
# ── Create the ontology item in Fabric ─────────────────────────────────
response = create_ontology_item(
    workspace_id=workspace_id,
    access_token=access_token,
    ontology_item_name=ontology_item_name,
    ontology_definition=ontology_definition,
)

print(f"Status: {response.status_code}")
print(json.dumps(response.json(), indent=2))

### Ontology Bindings Reference

| Entity Type | Lakehouse Table | Primary Key |
|-------------|----------------|-------------|
| Nurse | dim_nurses | nurse_id |
| Patient | dim_patients | patient_id |
| HospitalUnit | dim_hospital_units | unit_id |
| DocumentationType | dim_documentation_types | doc_type_id |
| Medication | dim_medications | medication_id |
| Diagnosis | dim_diagnoses | diagnosis_id |

| Contextualization | KQL Table | Key Bindings |
|-------------------|-----------|-------------|
| DocumentationEvent | documentation_events | Nurse, Patient, DocumentationType |
| MedicationAdministration | medication_administration | Nurse, Patient, Medication |
| VitalSignsRecording | vital_signs | Nurse, Patient |
| AssessmentEvent | assessments | Nurse, Patient |
| EHRInteraction | ehr_interactions | Nurse |
| HandoffReport | handoff_reports | Nurse (outgoing), Nurse (incoming), Patient |

| Relationship | Source → Target | Derived From |
|-------------|----------------|-------------|
| assigned_to | Nurse → HospitalUnit | dim_nurses.unit_id |
| admitted_to | Patient → HospitalUnit | dim_patients (via encounters) |
| has_diagnosis | Patient → Diagnosis | dim_patients.primary_diagnosis_id |
| cares_for | Nurse → Patient | fact_shifts / fact_patient_encounters |
| prescribes_for | Medication → Diagnosis | clinical protocol mapping |